# Explore a training run

Load a finished run's config and best checkpoint, then inspect predictions.
Register the kernel once with `just kernel`.

In [ ]:
from pathlib import Path

import torch
from omegaconf import OmegaConf

from startml import ROOT_DIR
from startml.utils import instantiate
from startml.utils.checkpoint import load_model_from_run

runs = sorted((ROOT_DIR / "outputs" / "train" / "runs").iterdir())
run_dir = runs[-1]  # latest run
run_dir

In [ ]:
model, cfg = load_model_from_run(run_dir, which="best")
print(OmegaConf.to_yaml(cfg.model))

In [ ]:
datamodule = instantiate(cfg.data, _convert_="all")
datamodule.prepare_data()
datamodule.setup("test")
x, y = next(iter(datamodule.test_dataloader()))

with torch.inference_mode():
    probs = model(x).softmax(-1)
preds = probs.argmax(-1)
print(f"batch accuracy: {(preds == y).float().mean():.3f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, img, p, t in zip(axes.flat, x, preds, y):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"{p.item()} ({t.item()})", color="green" if p == t else "red")
    ax.axis("off")
plt.tight_layout()

In [ ]:
# Training curves from the CSV logger (if the run used `logger=csv` or `logger=many`).
import pandas as pd

csv = run_dir / "csv" / "metrics.csv"
if csv.exists():
    df = pd.read_csv(csv)
    df.groupby("epoch")[["train/loss_epoch", "val/loss", "val/acc"]].mean().plot(subplots=True, figsize=(8, 6))